In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

import optuna



In [ ]:
# Load CSV file

house = pd.read_csv("bengaluru_house_prices.csv")

house.head()

In [ ]:
house.shape
house.info()
house.duplicated().sum()
house.isnull().sum()
house.columns

In [ ]:
house["price"].describe()

In [ ]:
# y = target
y = house["price"]

# X = features
X = house.drop("price", axis=1)

print("Categorical Columns:")

print(
    X.select_dtypes(
        include="object"
    ).columns
)

In [ ]:
# Convert size to number

X["size"] = X["size"].str.extract(
    "(\d+)"
)

X["size"] = pd.to_numeric(
    X["size"],
    errors="coerce"
)

X.head()

In [ ]:
def convert_sqft(x):

    try:

        if "-" in str(x):

            values = str(x).split("-")

            return (
                float(values[0]) +
                float(values[1])
            ) / 2

        return float(x)

    except:

        return np.nan

In [ ]:
X["total_sqft"] = X["total_sqft"].apply(
    convert_sqft
)

X["total_sqft"].head()

In [ ]:

X = X.drop(
    ["society"],
    axis=1
)

X.head()

In [ ]:
print("Missing Values Before:")
print(X.isnull().sum())

# Numerical columns

X["size"] = X["size"].fillna(
    X["size"].median()
)

X["total_sqft"] = X["total_sqft"].fillna(
    X["total_sqft"].median()
)

X["bath"] = X["bath"].fillna(
    X["bath"].median()
)

X["balcony"] = X["balcony"].fillna(
    X["balcony"].median()
)

# Categorical columns

X["location"] = X["location"].fillna(
    X["location"].mode()[0]
)

X["area_type"] = X["area_type"].fillna(
    X["area_type"].mode()[0]
)

X["availability"] = X["availability"].fillna(
    X["availability"].mode()[0]
)

print("Missing Values After:")
print(X.isnull().sum())

In [ ]:
X = pd.get_dummies(
    X,
    drop_first=True
)

X.head()
print("Final Shape:")
print(X.shape)

print("\nMissing Values:")
print(X.isnull().sum().sum())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)

print("X_test:", X_test.shape)

print("y_train:", y_train.shape)

print("y_test:", y_test.shape)

In [ ]:
model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train
)

y_predict = model.predict(
    X_test
)

In [ ]:
r2 = r2_score(
    y_test,
    y_predict
)

mae = mean_absolute_error(
    y_test,
    y_predict
)

mse = mean_squared_error(
    y_test,
    y_predict
)

rmse = np.sqrt(mse)

print("========== Random Forest ==========")

print("R2 Score:", r2)

print("MAE:", mae)

print("MSE:", mse)

print("RMSE:", rmse)

In [ ]:
def objective(trial):

    model = RandomForestRegressor(

        n_estimators=trial.suggest_int(
            "n_estimators",
            100,
            500
        ),

        max_depth=trial.suggest_int(
            "max_depth",
            5,
            30
        ),

        min_samples_split=trial.suggest_int(
            "min_samples_split",
            2,
            10
        ),

        min_samples_leaf=trial.suggest_int(
            "min_samples_leaf",
            1,
            5
        ),

        max_features=trial.suggest_categorical(
            "max_features",
            ["sqrt", "log2", 1.0]
        ),

        random_state=42,

        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    prediction = model.predict(
        X_test
    )

    r2 = r2_score(
        y_test,
        prediction
    )

    return r2

In [ ]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=50
)

print("Best R2:")
print(study.best_value)

print("\nBest Parameters:")
print(study.best_params)

In [ ]:
optuna_model = RandomForestRegressor(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

optuna_model.fit(
    X_train,
    y_train
)

optuna_predict = optuna_model.predict(
    X_test
)

In [ ]:
optuna_r2 = r2_score(
    y_test,
    optuna_predict
)

optuna_mae = mean_absolute_error(
    y_test,
    optuna_predict
)

optuna_mse = mean_squared_error(
    y_test,
    optuna_predict
)

optuna_rmse = np.sqrt(
    optuna_mse
)


print("========== Optuna House Model ==========")

print("Test R2:", optuna_r2)

print("MAE:", optuna_mae)

print("MSE:", optuna_mse)

print("RMSE:", optuna_rmse)

In [ ]:
space = {

    "n_estimators": hp.quniform(
        "n_estimators",
        100,
        500,
        10
    ),

    "max_depth": hp.quniform(
        "max_depth",
        5,
        30,
        1
    ),

    "min_samples_split": hp.quniform(
        "min_samples_split",
        2,
        10,
        1
    ),

    "min_samples_leaf": hp.quniform(
        "min_samples_leaf",
        1,
        5,
        1
    ),

    "max_features": hp.choice(
        "max_features",
        ["sqrt", "log2", 1.0]
    )
}

In [ ]:
def hyperopt_objective(params):

    params["n_estimators"] = int(
        params["n_estimators"]
    )

    params["max_depth"] = int(
        params["max_depth"]
    )

    params["min_samples_split"] = int(
        params["min_samples_split"]
    )

    params["min_samples_leaf"] = int(
        params["min_samples_leaf"]
    )

    model = RandomForestRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    prediction = model.predict(
        X_test
    )

    r2 = r2_score(
        y_test,
        prediction
    )

    # Hyperopt tries to minimize loss
    return {
        "loss": -r2,
        "status": STATUS_OK
    }

In [ ]:
trials = Trials()

best_hyperopt = fmin(
    fn=hyperopt_objective,
    space=space,
    algo=tpe.suggest,
    max_evals=50,
    trials=trials,
    rstate=np.random.default_rng(42)
)

print("Best Hyperopt Parameters:")
print(best_hyperopt)

In [ ]:
max_features_options = [
    "sqrt",
    "log2",
    1.0
]

hyperopt_model = RandomForestRegressor(

    n_estimators=int(
        best_hyperopt["n_estimators"]
    ),

    max_depth=int(
        best_hyperopt["max_depth"]
    ),

    min_samples_split=int(
        best_hyperopt["min_samples_split"]
    ),

    min_samples_leaf=int(
        best_hyperopt["min_samples_leaf"]
    ),

    max_features=max_features_options[
        int(best_hyperopt["max_features"])
    ],

    random_state=42,

    n_jobs=-1
)

hyperopt_model.fit(
    X_train,
    y_train
)

hyperopt_predict = hyperopt_model.predict(
    X_test
)

In [ ]:
hyperopt_r2 = r2_score(
    y_test,
    hyperopt_predict
)

hyperopt_mae = mean_absolute_error(
    y_test,
    hyperopt_predict
)

hyperopt_mse = mean_squared_error(
    y_test,
    hyperopt_predict
)

hyperopt_rmse = np.sqrt(
    hyperopt_mse
)


print("========== Hyperopt House Model ==========")

print("Test R2:", hyperopt_r2)

print("MAE:", hyperopt_mae)

print("MSE:", hyperopt_mse)

print("RMSE:", hyperopt_rmse)

In [ ]:
results = pd.DataFrame({

    "Model": [
        "Random Forest",
        "Optuna",
        "Hyperopt"
    ],

    "R2": [
        r2,
        optuna_r2,
        hyperopt_r2
    ],

    "MAE": [
        mae,
        optuna_mae,
        hyperopt_mae
    ],

    "MSE": [
        mse,
        optuna_mse,
        hyperopt_mse
    ],

    "RMSE": [
        rmse,
        optuna_rmse,
        hyperopt_rmse
    ]
})

results

In [ ]:
best_model_index = results["R2"].idxmax()

best_model_name = results.loc[
    best_model_index,
    "Model"
]

best_r2 = results.loc[
    best_model_index,
    "R2"
]

best_mae = results.loc[
    best_model_index,
    "MAE"
]

best_mse = results.loc[
    best_model_index,
    "MSE"
]

best_rmse = results.loc[
    best_model_index,
    "RMSE"
]

print("========== BEST MODEL ==========")

print("Model:", best_model_name)

print("R2:", best_r2)

print("MAE:", best_mae)

print("MSE:", best_mse)

print("RMSE:", best_rmse)

In [ ]:
print("==========================================")
print("        BENGALURU HOUSE PRICE MODEL")
print("==========================================")

print("Best Model:", best_model_name)

print(
    "R2 Score:",
    round(best_r2, 4)
)

print(
    "MAE:",
    round(best_mae, 4)
)

print(
    "MSE:",
    round(best_mse, 4)
)

print(
    "RMSE:",
    round(best_rmse, 4)
)

print("==========================================")